# TemporalRec — Visual EDA of the Final 5-Core Dataset

Yes, graphs are useful here, just as they were in the freight-rate project. These graphs answer recommendation-specific questions: Are sequences long enough? Are item interactions very uneven? Are time gaps varied? Does behavior shift over calendar time?

This notebook only reads the final 5-core dataset and writes chart images. It does not change data.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 30)

PROJECT_ROOT = Path.cwd()
FIVE_CORE_RUN_TAG = "five_core_2026_10_13"  # Must match Notebook 03.
VISUAL_RUN_TAG = "visual_eda_2026_10_13"

INPUT_FILE = PROJECT_ROOT / "data" / "interim" / FIVE_CORE_RUN_TAG / "events_all_event_types_5core.csv"
REPORT_DIR = PROJECT_ROOT / "reports" / VISUAL_RUN_TAG
FIGURE_DIR = REPORT_DIR / "figures"

assert INPUT_FILE.exists(), f"Cannot find {INPUT_FILE}. Run Notebook 03 first or correct FIVE_CORE_RUN_TAG."
assert not REPORT_DIR.exists(), f"{REPORT_DIR} exists. Change VISUAL_RUN_TAG rather than overwriting old charts."
FIGURE_DIR.mkdir(parents=True)

print(f"Input:   {INPUT_FILE}")
print(f"Figures: {FIGURE_DIR}")

In [ ]:
DTYPES = {
    "timestamp": "int64",
    "visitorid": "int64",
    "event": "category",
    "itemid": "int64",
    "transactionid": "Int64",
}
events = pd.read_csv(INPUT_FILE, dtype=DTYPES)
assert events.duplicated(["timestamp", "visitorid", "event", "itemid", "transactionid"], keep=False).sum() == 0
assert events.duplicated(["visitorid", "timestamp"], keep=False).sum() == 0

events["event_datetime_utc"] = pd.to_datetime(events["timestamp"], unit="ms", utc=True)
events["event_date_utc"] = events["event_datetime_utc"].dt.date
print(f"Events: {len(events):,}; visitors: {events['visitorid'].nunique():,}; items: {events['itemid'].nunique():,}")

## 1. Event-type mix

This shows how much of the sequence signal comes from views, carts, and purchases. The y-axis is logarithmic because views are much more common.

In [ ]:
event_counts = events["event"].value_counts().reindex(["view", "addtocart", "transaction"], fill_value=0)
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(event_counts.index.astype(str), event_counts.values, color=["#4C78A8", "#F58518", "#54A24B"])
ax.set_yscale("log")
ax.set_xlabel("Event type")
ax.set_ylabel("Event count (log scale)")
ax.set_title("Event mix in final 5-core dataset")
for bar, value in zip(bars, event_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, value, f"{value:,}", ha="center", va="bottom", fontsize=9)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "01_event_type_mix.png", dpi=160, bbox_inches="tight")
plt.show()

## 2. Visitor sequence lengths

This tells us how many previous interactions a typical visitor gives the model. We show the 99th percentile so one unusually active visitor does not stretch the chart.

In [ ]:
user_counts = events.groupby("visitorid").size()
user_cap = int(user_counts.quantile(0.99))
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(user_counts.clip(upper=user_cap), bins=50, color="#4C78A8", edgecolor="white")
ax.axvline(user_counts.median(), color="#E45756", linewidth=2, label=f"Median: {user_counts.median():.0f}")
ax.axvline(user_counts.quantile(0.90), color="#54A24B", linewidth=2, label=f"90th percentile: {user_counts.quantile(0.90):.0f}")
ax.set_xlabel(f"Interactions per visitor (values above {user_cap} grouped at the right edge)")
ax.set_ylabel("Number of visitors")
ax.set_title("Visitor sequence-length distribution")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "02_visitor_sequence_lengths.png", dpi=160, bbox_inches="tight")
plt.show()

## 3. Item-popularity long tail

Recommendation data normally has a few popular items and many less-popular items. This chart shows how strong that long-tail pattern is.

In [ ]:
item_counts = events.groupby("itemid").size().sort_values(ascending=False).reset_index(drop=True)
rank = np.arange(1, len(item_counts) + 1)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(rank, item_counts.values, color="#7E57C2", linewidth=1)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Item popularity rank (log scale)")
ax.set_ylabel("Interactions per item (log scale)")
ax.set_title("Item-popularity long tail")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "03_item_popularity_long_tail.png", dpi=160, bbox_inches="tight")
plt.show()

## 4. Time-gap distribution

This is the most important chart for TiSASRec. If time gaps vary, a time-aware model has information that an order-only model does not explicitly receive.

In [ ]:
ordered = events.sort_values(["visitorid", "timestamp"], kind="stable")
gap_hours = ordered.groupby("visitorid")["timestamp"].diff().dropna() / 3_600_000
gap_upper = gap_hours.quantile(0.999)
gap_plot = gap_hours.loc[(gap_hours > 0) & (gap_hours <= gap_upper)]
bins = np.logspace(np.log10(gap_plot.min()), np.log10(gap_plot.max()), 60)
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(gap_plot, bins=bins, color="#F58518", edgecolor="white")
ax.set_xscale("log")
ax.set_xlabel("Hours since the visitor's previous interaction (log scale)")
ax.set_ylabel("Number of gaps")
ax.set_title("Within-visitor time gaps (top 0.1% excluded only for chart readability)")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "04_time_gap_distribution.png", dpi=160, bbox_inches="tight")
plt.show()

## 5. Events over calendar time

A chronological split may face behavior changes over time. This daily chart lets us see whether the data is reasonably continuous and whether there are sharp spikes or quiet periods.

In [ ]:
daily_events = events.groupby("event_date_utc").size()
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(pd.to_datetime(daily_events.index), daily_events.values, color="#4C78A8", linewidth=1.5)
ax.set_xlabel("Date (UTC)")
ax.set_ylabel("Events per day")
ax.set_title("Retailrocket interaction volume over time")
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "05_events_over_time.png", dpi=160, bbox_inches="tight")
plt.show()

## Save a numeric summary

The numbers behind the charts are saved for the decision and experiment logs.

In [ ]:
summary = pd.DataFrame([
    {"metric": "events", "value": len(events)},
    {"metric": "visitors", "value": events["visitorid"].nunique()},
    {"metric": "items", "value": events["itemid"].nunique()},
    {"metric": "median visitor sequence length", "value": user_counts.median()},
    {"metric": "p90 visitor sequence length", "value": user_counts.quantile(0.90)},
    {"metric": "maximum visitor sequence length", "value": user_counts.max()},
    {"metric": "median item interactions", "value": item_counts.median()},
    {"metric": "p90 item interactions", "value": item_counts.quantile(0.90)},
    {"metric": "median time gap hours", "value": gap_hours.median()},
    {"metric": "p90 time gap hours", "value": gap_hours.quantile(0.90)},
    {"metric": "p99 time gap hours", "value": gap_hours.quantile(0.99)},
    {"metric": "first event date UTC", "value": str(events["event_datetime_utc"].min())},
    {"metric": "last event date UTC", "value": str(events["event_datetime_utc"].max())},
])
summary.to_csv(REPORT_DIR / "visual_eda_summary.csv", index=False)
summary

## What to inspect

- The visitor-length chart helps us choose a later maximum sequence length.
- The long-tail chart explains why popularity baselines can be strong and why ranking every item is difficult.
- The time-gap chart supports or challenges the reason for testing TiSASRec.
- The daily chart helps us choose chronological train/validation/test boundaries without placing a split inside a major abnormal period.

Do not choose a model from these charts. They are evidence for preprocessing and evaluation decisions.